In [ ]:

import sys
import platform

print("Python version:", sys.version)
print("Platform:", platform.platform())

try:
    import torch
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch is not installed yet.")


Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch version: 2.11.0+cpu
CUDA available: False


In [ ]:

%pip install -q transformers accelerate sentencepiece streamlit pandas numpy scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 11.4 MB/s eta 0:00:00


In [ ]:

import torch
import transformers
import streamlit
import pandas
import sklearn

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Streamlit:", streamlit.__version__)
print("Pandas:", pandas.__version__)
print("Scikit-learn:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch: 2.11.0+cpu
Transformers: 5.18.0
Streamlit: 1.65.0
Pandas: 2.2.3
Scikit-learn: 1.6.1
CUDA available: False


In [ ]:

import pandas as pd

manufacturing_data = pd.DataFrame([
    {
        "topic": "High vibration",
        "symptom": "Vibration is above the normal operating range",
        "possible_cause": "Imbalance, bearing wear, or misalignment",
        "recommended_action": "Check vibration trends, inspect mounting and bearings, and follow the approved maintenance procedure"
    },
    {
        "topic": "High temperature",
        "symptom": "Machine temperature is rising",
        "possible_cause": "Overheating, poor cooling, excessive load, or friction",
        "recommended_action": "Verify the temperature sensor, inspect cooling, and check the machine load"
    },
    {
        "topic": "Low pressure",
        "symptom": "Pressure is below the expected operating range",
        "possible_cause": "Leakage, blockage, or pump performance issues",
        "recommended_action": "Check the pressure sensor, inspect for leaks, and follow the equipment procedure"
    },
    {
        "topic": "Voltage fluctuation",
        "symptom": "Voltage readings are unstable",
        "possible_cause": "Power supply variation, loose connections, or sensor issues",
        "recommended_action": "Ask a qualified technician to inspect the electrical supply and connections"
    },
    {
        "topic": "Machine failure",
        "symptom": "Failure event or machine stoppage is recorded",
        "possible_cause": "Mechanical, electrical, control, or process fault",
        "recommended_action": "Review recent sensor readings and error logs, follow lockout procedures where applicable, and have authorized personnel inspect the machine"
    }
])

print("Dataset shape:", manufacturing_data.shape)
display(manufacturing_data)


Dataset shape: (5, 4)


,topic,symptom,possible_cause,recommended_action
0,High vibration,Vibration is above the normal operating range,"Imbalance, bearing wear, or misalignment","Check vibration trends, inspect mounting and b..."
1,High temperature,Machine temperature is rising,"Overheating, poor cooling, excessive load, or ...","Verify the temperature sensor, inspect cooling..."
2,Low pressure,Pressure is below the expected operating range,"Leakage, blockage, or pump performance issues","Check the pressure sensor, inspect for leaks, ..."
3,Voltage fluctuation,Voltage readings are unstable,"Power supply variation, loose connections, or ...",Ask a qualified technician to inspect the elec...
4,Machine failure,Failure event or machine stoppage is recorded,"Mechanical, electrical, control, or process fault","Review recent sensor readings and error logs, ..."


In [ ]:

from pathlib import Path

DATA_DIR = Path("/content/manufacturing_slm")
DATA_DIR.mkdir(parents=True, exist_ok=True)

knowledge_path = DATA_DIR / "manufacturing_knowledge.csv"
manufacturing_data.to_csv(knowledge_path, index=False)

print("Knowledge file saved:", knowledge_path)
print("Records saved:", len(manufacturing_data))
print("Columns:", manufacturing_data.columns.tolist())


Knowledge file saved: /content/manufacturing_slm/manufacturing_knowledge.csv
Records saved: 5
Columns: ['topic', 'symptom', 'possible_cause', 'recommended_action']


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = torch.device("cpu")

print("Model:", MODEL_ID)
print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())


Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cpu
CUDA available: False


In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32
)

model.to(DEVICE)
model.eval()

print("SLM loaded successfully on CPU.")


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

SLM loaded successfully on CPU.


In [ ]:

messages = [
    {
        "role": "system",
        "content": "You are a manufacturing maintenance assistant. Give concise, safety-conscious answers."
    },
    {
        "role": "user",
        "content": "What should an operator check when machine vibration increases?"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("SLM response:")
print(answer)


SLM response:
An operator should immediately stop the machine and inspect for any signs of damage or unusual vibrations. They should also ensure that all necessary safety measures have been taken to prevent further injury. If the issue persists, it may be necessary to consult with a professional mechanic or engineer for further assistance.


In [ ]:

import pandas as pd
from pathlib import Path

knowledge_path = Path("/content/manufacturing_slm/manufacturing_knowledge.csv")
knowledge_df = pd.read_csv(knowledge_path)

def find_manufacturing_context(question):
    question = question.lower()

    keywords = {
        "vibration": ["vibration", "bearing", "imbalance", "misalignment"],
        "temperature": ["temperature", "overheat", "hot", "cooling"],
        "pressure": ["pressure", "leak", "pump"],
        "voltage": ["voltage", "electrical", "power supply"],
        "failure": ["failure", "stoppage", "stopped", "machine fault"]
    }

    matched_topics = []
    for topic, terms in keywords.items():
        if any(term in question for term in terms):
            matched_topics.append(topic)

    if not matched_topics:
        return "No directly matching record was found in the example knowledge base."

    matches = knowledge_df[
        knowledge_df["topic"].str.lower().isin(matched_topics)
    ]

    if matches.empty:
        return "No directly matching record was found in the example knowledge base."

    return "\n".join(
        f"Topic: {row['topic']}\n"
        f"Symptom: {row['symptom']}\n"
        f"Possible causes: {row['possible_cause']}\n"
        f"Suggested checks: {row['recommended_action']}"
        for _, row in matches.iterrows()
    )

print(find_manufacturing_context(
    "What should I check if machine vibration increases?"
))


No directly matching record was found in the example knowledge base.


In [ ]:

def find_manufacturing_context(question):
    question = question.lower()

    keywords = {
        "High vibration": [
            "vibration", "bearing", "imbalance", "misalignment"
        ],
        "High temperature": [
            "temperature", "overheat", "overheating", "hot", "cooling"
        ],
        "Low pressure": [
            "pressure", "leak", "leakage", "pump"
        ],
        "Voltage fluctuation": [
            "voltage", "electrical", "power supply"
        ],
        "Machine failure": [
            "failure", "stoppage", "stopped", "machine fault"
        ]
    }

    matched_topics = [
        topic for topic, terms in keywords.items()
        if any(term in question for term in terms)
    ]

    matches = knowledge_df[
        knowledge_df["topic"].isin(matched_topics)
    ]

    if matches.empty:
        return "No matching record found in the example knowledge base."

    results = []
    for _, row in matches.iterrows():
        results.append(
            f"Topic: {row['topic']}\n"
            f"Symptom: {row['symptom']}\n"
            f"Possible causes: {row['possible_cause']}\n"
            f"Suggested checks: {row['recommended_action']}"
        )

    return "\n\n".join(results)


print(find_manufacturing_context(
    "What should I check if machine vibration increases?"
))


Topic: High vibration
Symptom: Vibration is above the normal operating range
Possible causes: Imbalance, bearing wear, or misalignment
Suggested checks: Check vibration trends, inspect mounting and bearings, and follow the approved maintenance procedure


In [ ]:

def ask_manufacturing_slm(question):
    context = find_manufacturing_context(question)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a manufacturing maintenance assistant. "
                "Answer clearly using the supplied manufacturing context. "
                "Do not invent sensor readings or claim to know the exact cause "
                "of a fault. If context is insufficient, say so. "
                "Recommend following approved site safety procedures."
            )
        },
        {
            "role": "user",
            "content": (
                f"Manufacturing knowledge:\n{context}\n\n"
                f"Operator question: {question}\n\n"
                "Give a concise answer with possible causes and suggested checks."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


answer = ask_manufacturing_slm(
    "What should I check if machine vibration increases?"
)

print("Manufacturing SLM Answer:")
print(answer)


Manufacturing SLM Answer:
If machine vibration increases, you should first check for any signs of imbalance or misalignment in the equipment. Next, inspect the mounting points and bearings for any signs of wear or damage. Finally, review the operator's manual for specific recommendations on how to address the issue.


In [ ]:

from pathlib import Path

app_code = r'''
import streamlit as st
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path

st.set_page_config(
    page_title="Manufacturing SLM Intelligence",
    page_icon="🏭",
    layout="wide"
)

st.title("🏭 Manufacturing SLM Intelligence")
st.write("A CPU-based language model for manufacturing maintenance assistance.")

st.warning(
    "Prototype demonstration using example knowledge. "
    "Always follow approved machine safety procedures."
)

DATA_PATH = Path("/content/manufacturing_slm/manufacturing_knowledge.csv")
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

@st.cache_resource
def load_assistant():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32
    )
    model.to("cpu")
    model.eval()
    knowledge = pd.read_csv(DATA_PATH)
    return tokenizer, model, knowledge

def retrieve_context(question, knowledge):
    q = question.lower()
    rules = {
        "High vibration": ["vibration", "bearing", "imbalance", "misalignment"],
        "High temperature": ["temperature", "overheat", "hot", "cooling"],
        "Low pressure": ["pressure", "leak", "pump"],
        "Voltage fluctuation": ["voltage", "electrical", "power supply"],
        "Machine failure": ["failure", "stoppage", "stopped", "machine fault"]
    }

    topics = [
        topic for topic, words in rules.items()
        if any(word in q for word in words)
    ]
    matches = knowledge[knowledge["topic"].isin(topics)]

    if matches.empty:
        return "No matching record found in the example knowledge base."

    return "\n".join(
        f"Topic: {r['topic']}; Possible causes: {r['possible_cause']}; "
        f"Suggested checks: {r['recommended_action']}"
        for _, r in matches.iterrows()
    )

def generate_answer(question, context, tokenizer, model):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise manufacturing maintenance assistant. "
                "Use the supplied context. Do not invent readings or claim "
                "a definitive fault diagnosis. If information is insufficient, "
                "say so. Follow approved site safety procedures."
            )
        },
        {
            "role": "user",
            "content": f"Knowledge:\n{context}\n\nQuestion: {question}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

with st.spinner("Loading the SLM on CPU..."):
    try:
        tokenizer, model, knowledge = load_assistant()
        st.success("SLM loaded on CPU.")
    except Exception as e:
        st.error(f"Could not load the model or knowledge file: {e}")
        st.stop()

st.subheader("Ask the Manufacturing Assistant")
question = st.text_area(
    "Enter an operator question",
    placeholder="Example: What should I check if machine vibration increases?"
)

if st.button("Get answer", type="primary"):
    if not question.strip():
        st.warning("Please enter a question.")
    else:
        context = retrieve_context(question, knowledge)
        with st.spinner("Generating answer on CPU..."):
            answer = generate_answer(question, context, tokenizer, model)

        st.markdown("### Assistant answer")
        st.write(answer)

        with st.expander("View retrieved manufacturing context"):
            st.write(context)

st.divider()
st.caption("CPU inference • Example manufacturing knowledge • Human review required")
'''

Path("/content/manufacturing_slm/app.py").write_text(
    app_code, encoding="utf-8"
)
print("Application file created:", Path("/content/manufacturing_slm/app.py").exists())


Application file created: True


In [ ]:

import ast
from pathlib import Path

app_path = Path("/content/manufacturing_slm/app.py")
ast.parse(app_path.read_text(encoding="utf-8"))

print("Application syntax: OK")
print("File:", app_path)


Application syntax: OK
File: /content/manufacturing_slm/app.py


In [ ]:
!python -m streamlit run /content/manufacturing_slm/app.py --server.port 8501 --server.address 0.0.0.0 > /content/manufacturing_slm/streamlit.log 2>&1 &

In [ ]:
import time
import requests

time.sleep(5)

try:
    response = requests.get(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=10
    )
    print("Status:", response.status_code)
    print("Response:", response.text)
except Exception as e:
    print("Connection error:", e)
    print(open("/content/manufacturing_slm/streamlit.log").read()[-3000:])

Status: 200
Response: ok


In [ ]:
import requests

try:
    response = requests.get(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=10
    )
    print("Status:", response.status_code)
    print("Response:", response.text)
except Exception as e:
    print("Connection error:", e)

print("\n--- Recent application log ---")
from pathlib import Path
log_path = Path("/content/manufacturing_slm/streamlit.log")
print(log_path.read_text()[-3000:] if log_path.exists() else "Log file not found")

Status: 200
Response: ok

--- Recent application log ---


2026-10-14 11:25:29.550 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.171.44.70:8501




In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501, height=800)

<IPython.core.display.Javascript object>

In [ ]:
import requests

r = requests.get("http://127.0.0.1:8501", timeout=15)

print("Page status:", r.status_code)
print("Page size:", len(r.text))
print("Contains Streamlit:", "streamlit" in r.text.lower())
print(r.text[:500])

Page status: 200
Page size: 6463
Contains Streamlit: True
<!--
 Copyright (c) Streamlit Inc. (2018-2022) Snowflake Inc. (2022-2026)

 Licensed under the Apache License, Version 2.0 (the "License");
 you may not use this file except in compliance with the License.
 You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing, software
 distributed under the License is distributed on an "AS IS" BASIS,
 WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or im


In [ ]:
from pathlib import Path

print(
    Path("/content/manufacturing_slm/streamlit.log")
    .read_text()[-5000:]
)

07
2026-10-14 11:31:09.385 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-usc1b0-1g0z8sf68cxcc-b.us-central1-0.prod.colab.dev, host=m-s-kkb-usc1b0-1g0z8sf68cxcc.us-central1-b.c.codatalab-user-runtimes.internal:8007
2026-10-14 11:31:10.441 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-usc1b0-1g0z8sf68cxcc-b.us-central1-0.prod.colab.dev, host=m-s-kkb-usc1b0-1g0z8sf68cxcc.us-central1-b.c.codatalab-user-runtimes.internal:8007
2026-10-14 11:31:14.678 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-usc1b0-1g0z8sf68cxcc-b.us-central1-0.prod.colab.dev, host=m-s-kkb-usc1b0-1g0z8sf68cxcc.us-central1-b.c.codatalab-user-runtimes.internal:8007
2026-10-14 11:31:17.584 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-usc1b0-1g0z8sf68cxcc-b.us-central1-0.prod.colab.dev, host=m-s-kkb-usc1b0-1g0z8sf68cxcc

In [ ]:
!pkill -f "streamlit run /content/manufacturing_slm/app.py" || true

^C


In [ ]:
!streamlit run /content/manufacturing_slm/app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.enableWebsocketCompression false \
  --server.allowedHosts "*" \
  > /content/manufacturing_slm/streamlit.log 2>&1 &

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501, height=850)

<IPython.core.display.Javascript object>

In [ ]:
!tail -n 40 /content/manufacturing_slm/streamlit.log



2026-10-14 11:33:34.232 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.171.44.70:8501

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:03<00:00, 76.60it/s]


In [ ]:
!ps aux | grep '[s]treamlit run'

root        7507 19.2 20.0 4677340 2662456 ?     Sl   11:33   0:26 /usr/bin/python3 /usr/local/bin/streamlit run /content/manufacturing_slm/app.py --server.port 8501 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false --server.enableWebsocketCompression false --server.allowedHosts *


In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
import requests

r = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=10)
print("Status:", r.status_code)
print("Response:", r.text)

Status: 200
Response: ok


In [ ]:
from pathlib import Path

print("App file exists:", Path("/content/manufacturing_slm/app.py").exists())
print("Knowledge file exists:", Path("/content/manufacturing_slm/manufacturing_knowledge.csv").exists())

App file exists: True
Knowledge file exists: True


In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501, height=850)

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path

app = Path("/content/manufacturing_slm/app.py")
data = Path("/content/manufacturing_slm/manufacturing_knowledge.csv")

print("App file:", app.exists())
print("Knowledge CSV:", data.exists())
print("Streamlit log:")
print(Path("/content/manufacturing_slm/streamlit.log").read_text()[-1500:])

App file: True
Knowledge CSV: True
Streamlit log:


2026-10-14 11:33:34.232 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.171.44.70:8501

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

Loading weights: 100%|██████████| 290/290 [00:03<00:00, 76.60it/s]



In [ ]:
import requests

for path in ["/_stcore/health", "/"]:
    try:
        r = requests.get("http://127.0.0.1:8501" + path, timeout=10)
        print(path, "Status:", r.status_code, "Size:", len(r.text))
        print(r.text[:100] if path == "/_stcore/health" else "Main page received")
    except Exception as e:
        print(path, "Error:", e)

/_stcore/health Status: 200 Size: 2
ok
/ Status: 200 Size: 6463
Main page received


In [ ]:
from google.colab import output

print("Opening Manufacturing SLM on port 8501...")
output.serve_kernel_port_as_window(8501)

Opening Manufacturing SLM on port 8501...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
!grep -Ei "error|exception|websocket|traceback" /content/manufacturing_slm/streamlit.log | tail -n 20

In [ ]:
!pkill -f "streamlit run /content/manufacturing_slm/app.py" || true

^C


In [ ]:
!streamlit run /content/manufacturing_slm/app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.enableWebsocketCompression false \
  --server.allowedHosts "*" \
  --browser.gatherUsageStats false \
  > /content/manufacturing_slm/streamlit.log 2>&1 &

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501, height=850)

<IPython.core.display.Javascript object>

In [ ]:
import shutil

zip_path = shutil.make_archive(
    "/content/Manufacturing_SLM_CPU_App",
    "zip",
    "/content/manufacturing_slm"
)

print("Project backup created:", zip_path)

Project backup created: /content/Manufacturing_SLM_CPU_App.zip


In [ ]:
from pathlib import Path
import subprocess

project = Path("/content/manufacturing_slm")

print("App exists:", (project / "app.py").exists())
print("Knowledge base exists:", (project / "manufacturing_knowledge.csv").exists())

subprocess.run(
    ["pkill", "-f", "streamlit run /content/manufacturing_slm/app.py"],
    check=False
)

App exists: True
Knowledge base exists: True


CompletedProcess(args=['pkill', '-f', 'streamlit run /content/manufacturing_slm/app.py'], returncode=0)

In [ ]:
from pathlib import Path
import shutil

project = Path("/content/manufacturing_slm")
app_file = project / "app.py"

# Keep a backup of the current application
if app_file.exists():
    shutil.copy2(app_file, project / "app_backup.py")

app_code = r'''
import streamlit as st
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from transformers import AutoTokenizer, AutoModelForCausalLM

PROJECT = Path("/content/manufacturing_slm")
CSV_FILE = PROJECT / "manufacturing_knowledge.csv"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

st.set_page_config(page_title="Manufacturing SLM", page_icon="🏭", layout="wide")
st.title("🏭 Manufacturing SLM Intelligence")
st.caption("CPU-based manufacturing assistant with image upload and sensor monitoring")

st.warning(
    "Prototype only. Sensor limits below are configurable examples, "
    "not certified safety limits. Follow approved plant procedures."
)

@st.cache_resource
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float32
    )
    model.to("cpu")
    model.eval()
    return tokenizer, model

@st.cache_data
def load_knowledge():
    if CSV_FILE.exists():
        return pd.read_csv(CSV_FILE)
    return pd.DataFrame(columns=[
        "topic", "symptom", "possible_cause", "recommended_action"
    ])

knowledge_df = load_knowledge()

def find_context(question):
    q = question.lower()
    keyword_map = {
        "High vibration": ["vibration", "bearing", "imbalance", "misalignment"],
        "High temperature": ["temperature", "overheat", "overheating", "hot", "cooling"],
        "Low pressure": ["pressure", "leak", "leakage", "pump"],
        "Voltage fluctuation": ["voltage", "electrical", "power supply"],
        "Machine failure": ["failure", "stoppage", "stopped", "machine fault"]
    }

    if knowledge_df.empty:
        return "No knowledge-base records are available."

    matched = [
        topic for topic, words in keyword_map.items()
        if any(word in q for word in words)
    ]

    if "topic" not in knowledge_df.columns:
        return "Knowledge-base CSV does not contain a topic column."

    rows = knowledge_df[knowledge_df["topic"].isin(matched)]
    if rows.empty:
        return "No matching record found in the example knowledge base."

    return "\n\n".join(
        "\n".join(f"{col}: {row[col]}" for col in rows.columns)
        for _, row in rows.iterrows()
    )

def generate_answer(question, sensor_summary, image_note):
    tokenizer, model = load_model()
    context = find_context(question)

    prompt = f"""
You are a manufacturing maintenance assistant.
Give concise, cautious guidance to a factory operator.
Use the supplied knowledge and sensor information.
Do not invent measurements or claim a definite diagnosis.
Treat uploaded images as unanalysed unless image observations are supplied.
Recommend following approved site procedures and escalating unsafe conditions.

Manufacturing knowledge:
{context}

Sensor findings:
{sensor_summary}

Image status:
{image_note}

Operator question:
{question}
"""
    messages = [
        {"role": "system", "content": "You are a careful manufacturing assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=220, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

left, right = st.columns([1, 1])

with left:
    st.subheader("📷 Machine image")
    uploaded_image = st.file_uploader(
        "Upload a machine photo", type=["png", "jpg", "jpeg"]
    )
    image_note = "No image uploaded."
    if uploaded_image is not None:
        try:
            image = Image.open(uploaded_image).convert("RGB")
            st.image(image, caption="Uploaded machine image", use_container_width=True)
            image_note = (
                "Image uploaded and displayed. No automated visual defect "
                "analysis has been performed."
            )
        except Exception as exc:
            st.error(f"Could not open image: {exc}")
            image_note = "Image could not be opened."

with right:
    st.subheader("📈 Sensor readings")
    st.caption("Enter your measurements and adjust the example thresholds to match approved limits.")
    c1, c2 = st.columns(2)

    with c1:
        vibration = st.number_input("Vibration (mm/s)", min_value=0.0, value=2.0, step=0.1)
        vibration_limit = st.number_input("Vibration example limit", min_value=0.1, value=4.5, step=0.1)
        temperature = st.number_input("Temperature (°C)", value=60.0, step=1.0)
        temperature_limit = st.number_input("Temperature example limit", value=80.0, step=1.0)

    with c2:
        pressure = st.number_input("Pressure (bar)", min_value=0.0, value=5.0, step=0.1)
        pressure_min = st.number_input("Minimum pressure example", min_value=0.0, value=3.0, step=0.1)
        voltage = st.number_input("Voltage (V)", min_value=0.0, value=230.0, step=1.0)
        voltage_min = st.number_input("Minimum voltage example", min_value=0.0, value=210.0, step=1.0)
        voltage_max = st.number_input("Maximum voltage example", min_value=0.1, value=250.0, step=1.0)

sensor_values = {
    "Vibration": (vibration, f"mm/s; example upper limit {vibration_limit}", vibration > vibration_limit),
    "Temperature": (temperature, f"°C; example upper limit {temperature_limit}", temperature > temperature_limit),
    "Pressure": (pressure, f"bar; example minimum {pressure_min}", pressure < pressure_min),
    "Voltage": (voltage, f"V; example range {voltage_min}–{voltage_max}", voltage < voltage_min or voltage > voltage_max)
}

findings = []
for name, (value, description, flagged) in sensor_values.items():
    if flagged:
        findings.append(f"{name}: {value} ({description}) — OUTSIDE EXAMPLE LIMIT")
    else:
        findings.append(f"{name}: {value} ({description}) — within example limits")

sensor_summary = "\n".join(findings)

st.subheader("Sensor status")
for name, (value, description, flagged) in sensor_values.items():
    if flagged:
        st.warning(f"{name}: {value} — outside the configured example limit")
    else:
        st.success(f"{name}: {value} — within the configured example limit")

st.subheader("💬 Ask the manufacturing assistant")
question = st.text_area(
    "Your question",
    value="Review these sensor readings and suggest what I should check.",
    height=100
)

if st.button("Analyze readings and answer", type="primary"):
    if not question.strip():
        st.error("Please enter a question.")
    else:
        with st.spinner("Generating answer on CPU..."):
            try:
                answer = generate_answer(question, sensor_summary, image_note)
                st.subheader("Assistant answer")
                st.write(answer)
                with st.expander("Show sensor findings and retrieved knowledge"):
                    st.text(sensor_summary)
                    st.text(find_context(question))
                if uploaded_image is not None:
                    st.info(
                        "The image is displayed for operator review only. "
                        "Automated image defect detection is not yet connected."
                    )
            except Exception as exc:
                st.error(f"Could not generate an answer: {exc}")
'''

app_file.write_text(app_code, encoding="utf-8")
print("Upgraded app saved:", app_file)
print("Backup saved:", project / "app_backup.py")

Upgraded app saved: /content/manufacturing_slm/app.py
Backup saved: /content/manufacturing_slm/app_backup.py


In [ ]:
!pkill -f "streamlit run /content/manufacturing_slm/app.py" || true

^C


In [ ]:
!streamlit run /content/manufacturing_slm/app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.enableWebsocketCompression false \
  --server.allowedHosts "*" \
  > /content/manufacturing_slm/streamlit.log 2>&1 &

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501, height=1000)

<IPython.core.display.Javascript object>

In [ ]:
%pip install -q ucimlrepo

from ucimlrepo import fetch_ucirepo
from pathlib import Path
import pandas as pd

project = Path("/content/manufacturing_slm")
project.mkdir(parents=True, exist_ok=True)

dataset = fetch_ucirepo(id=601)

X = dataset.data.features
y = dataset.data.targets

df = pd.concat([X, y], axis=1)
csv_path = project / "ai4i2020.csv"
df.to_csv(csv_path, index=False)

print("Dataset saved:", csv_path)
print("Rows and columns:", df.shape)
print(df.head())
print("Column names:", df.columns.tolist())

Dataset saved: /content/manufacturing_slm/ai4i2020.csv
Rows and columns: (10000, 12)
  Type  Air temperature  Process temperature  Rotational speed  Torque  \
0    M            298.1                308.6              1551    42.8   
1    L            298.2                308.7              1408    46.3   
2    L            298.1                308.5              1498    49.4   
3    L            298.2                308.6              1433    39.5   
4    L            298.2                308.7              1408    40.0   

   Tool wear  Machine failure  TWF  HDF  PWF  OSF  RNF  
0          0                0    0    0    0    0    0  
1          3                0    0    0    0    0    0  
2          5                0    0    0    0    0    0  
3          7                0    0    0    0    0    0  
4          9                0    0    0    0    0    0  
Column names: ['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Machine failure', '

In [ ]:
from pathlib import Path
import pandas as pd

csv_path = Path("/content/manufacturing_slm/ai4i2020.csv")

print("File exists:", csv_path.exists())

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("Dataset shape:", df.shape)
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nFirst 5 rows:")
    display(df.head())
    print("\nMissing values:", int(df.isnull().sum().sum()))

File exists: True
Dataset shape: (10000, 12)

Columns:
['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

First 5 rows:


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0



Missing values: 0


In [56]:
import pandas as pd

df = pd.read_csv("/content/manufacturing_slm/ai4i2020.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFailure counts:")
print(df["Machine failure"].value_counts() if "Machine failure" in df.columns else df.iloc[:, -1].value_counts())

Dataset shape: (10000, 12)
Columns: ['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

Failure counts:
Machine failure
0    9661
1     339
Name: count, dtype: int64


In [55]:
import pandas as pd

df = pd.read_csv("/content/manufacturing_slm/ai4i2020.csv")

print("Dataset information:")
df.info()

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nMachine type counts:")
print(df["Type"].value_counts())

print("\nFailure-mode counts:")
print(df[["TWF", "HDF", "PWF", "OSF", "RNF"]].sum())

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Type                 10000 non-null  object 
 1   Air temperature      10000 non-null  float64
 2   Process temperature  10000 non-null  float64
 3   Rotational speed     10000 non-null  int64  
 4   Torque               10000 non-null  float64
 5   Tool wear            10000 non-null  int64  
 6   Machine failure      10000 non-null  int64  
 7   TWF                  10000 non-null  int64  
 8   HDF                  10000 non-null  int64  
 9   PWF                  10000 non-null  int64  
 10  OSF                  10000 non-null  int64  
 11  RNF                  10000 non-null  int64  
dtypes: float64(3), int64(8), object(1)
memory usage: 937.6+ KB

Missing values in each column:
Type                   0
Air temperature        0
Process temperature    0


In [54]:
import pandas as pd

DATA_PATH = "/content/manufacturing_slm/ai4i2020.csv"

df = pd.read_csv(DATA_PATH)

failure_count = int(df["Machine failure"].sum())
total_records = len(df)
normal_count = total_records - failure_count

print("Manufacturing Dataset Summary")
print("-" * 35)
print(f"Total records: {total_records:,}")
print(f"Normal records: {normal_count:,}")
print(f"Machine failures: {failure_count:,}")
print(f"Failure rate: {failure_count / total_records * 100:.2f}%")

print("\nAverage sensor readings:")
print(df[[
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear"
]].mean().round(2))

print("\nFailure-mode counts:")
print(df[["TWF", "HDF", "PWF", "OSF", "RNF"]].sum().astype(int))

Manufacturing Dataset Summary
-----------------------------------
Total records: 10,000
Normal records: 9,661
Machine failures: 339
Failure rate: 3.39%

Average sensor readings:
Air temperature         300.00
Process temperature     310.01
Rotational speed       1538.78
Torque                   39.99
Tool wear               107.95
dtype: float64

Failure-mode counts:
TWF     46
HDF    115
PWF     95
OSF     98
RNF     19
dtype: int64


In [57]:
print("Colab is working!")

import pandas as pd
print("Pandas is working!")

df = pd.read_csv("/content/manufacturing_slm/ai4i2020.csv")
print("Dataset loaded:", df.shape)

Colab is working!
Pandas is working!
Dataset loaded: (10000, 12)


In [59]:
import pandas as pd

df = pd.read_csv("/content/manufacturing_slm/ai4i2020.csv")

print("Dataset loaded successfully:", df.shape)
print("Machine failures:", int(df["Machine failure"].sum()))
print("Normal records:", int((df["Machine failure"] == 0).sum()))

Dataset loaded successfully: (10000, 12)
Machine failures: 339
Normal records: 9661


In [60]:
from pathlib import Path

project = Path("/content/manufacturing_slm")

for filename in ["app.py", "app_backup.py", "ai4i2020.csv"]:
    path = project / filename
    print(f"{filename}: {'Found' if path.exists() else 'Missing'}")

app.py: Found
app_backup.py: Found
ai4i2020.csv: Found


In [61]:
from pathlib import Path
import shutil

project = Path("/content/manufacturing_slm")

shutil.copy2(project / "app.py", project / "app_before_analytics.py")

print("Backup created successfully!")
print("Your original app.py is unchanged.")

Backup created successfully!
Your original app.py is unchanged.


In [63]:
from pathlib import Path

project = Path("/content/manufacturing_slm")

analytics_code = '''
import streamlit as st
import pandas as pd

@st.cache_data
def load_ai4i_dataset():
    return pd.read_csv("/content/manufacturing_slm/ai4i2020.csv")

def show_dataset_analytics():
    st.header("AI4I Predictive Maintenance Dataset")

    df = load_ai4i_dataset()

    total = len(df)
    failures = int(df["Machine failure"].sum())
    normal = total - failures

    c1, c2, c3 = st.columns(3)
    c1.metric("Total records", f"{total:,}")
    c2.metric("Normal records", f"{normal:,}")
    c3.metric("Failure records", f"{failures:,}")

    st.metric("Dataset failure rate", f"{failures / total * 100:.2f}%")

    st.subheader("Failure modes")
    modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]
    mode_counts = df[modes].sum().astype(int)
    st.bar_chart(mode_counts)

    st.subheader("Sensor readings")
    sensor_cols = [
        "Air temperature",
        "Process temperature",
        "Rotational speed",
        "Torque",
        "Tool wear"
    ]

    selected_sensor = st.selectbox("Choose a sensor", sensor_cols)
    st.line_chart(df[selected_sensor].head(200))

    st.subheader("Dataset preview")
    st.dataframe(df.head(20), use_container_width=True)
'''

(project / "dataset_analytics.py").write_text(analytics_code)

print("Created:", project / "dataset_analytics.py")
print("Your existing app.py has not been changed.")

Created: /content/manufacturing_slm/dataset_analytics.py
Your existing app.py has not been changed.


In [64]:
from pathlib import Path

app_path = Path("/content/manufacturing_slm/app.py")
app_text = app_path.read_text()

print("App file length:", len(app_text), "characters")
print("Has Streamlit tabs:", "st.tabs(" in app_text)
print("Has question input:", "text_area" in app_text)
print("\nFirst 20 lines:")
print("\n".join(app_text.splitlines()[:20]))

App file length: 7459 characters
Has Streamlit tabs: False
Has question input: True

First 20 lines:

import streamlit as st
import pandas as pd
import torch
from pathlib import Path
from PIL import Image
from transformers import AutoTokenizer, AutoModelForCausalLM

PROJECT = Path("/content/manufacturing_slm")
CSV_FILE = PROJECT / "manufacturing_knowledge.csv"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

st.set_page_config(page_title="Manufacturing SLM", page_icon="🏭", layout="wide")
st.title("🏭 Manufacturing SLM Intelligence")
st.caption("CPU-based manufacturing assistant with image upload and sensor monitoring")

st.warning(
    "Prototype only. Sensor limits below are configurable examples, "
    "not certified safety limits. Follow approved plant procedures."
)


In [65]:
from pathlib import Path

app_path = Path("/content/manufacturing_slm/app.py")
lines = app_path.read_text().splitlines()

print("\n".join(
    f"{i + 1}: {line}"
    for i, line in enumerate(lines[-45:])
))

1:     "Voltage": (voltage, f"V; example range {voltage_min}–{voltage_max}", voltage < voltage_min or voltage > voltage_max)
2: }
3: 
4: findings = []
5: for name, (value, description, flagged) in sensor_values.items():
6:     if flagged:
7:         findings.append(f"{name}: {value} ({description}) — OUTSIDE EXAMPLE LIMIT")
8:     else:
9:         findings.append(f"{name}: {value} ({description}) — within example limits")
10: 
11: sensor_summary = "\n".join(findings)
12: 
13: st.subheader("Sensor status")
14: for name, (value, description, flagged) in sensor_values.items():
15:     if flagged:
16:         st.warning(f"{name}: {value} — outside the configured example limit")
17:     else:
18:         st.success(f"{name}: {value} — within the configured example limit")
19: 
20: st.subheader("💬 Ask the manufacturing assistant")
21: question = st.text_area(
22:     "Your question",
23:     value="Review these sensor readings and suggest what I should check.",
24:     height=100
25: )
26: 


In [66]:
from pathlib import Path

app_path = Path("/content/manufacturing_slm/app.py")
app_text = app_path.read_text()

if "AI4I Dataset Analytics" in app_text:
    print("Analytics section already exists. No changes made.")
else:
    analytics_section = '''

# --------------------------------------------------
# AI4I 2020 DATASET ANALYTICS
# --------------------------------------------------
st.divider()
st.header("📊 AI4I Dataset Analytics")

DATASET_PATH = PROJECT / "ai4i2020.csv"

if DATASET_PATH.exists():
    @st.cache_data
    def load_ai4i_data(path):
        return pd.read_csv(path)

    ai4i_df = load_ai4i_data(str(DATASET_PATH))

    total = len(ai4i_df)
    failures = int(ai4i_df["Machine failure"].sum())
    normal = total - failures

    col1, col2, col3 = st.columns(3)
    col1.metric("Total records", f"{total:,}")
    col2.metric("Normal records", f"{normal:,}")
    col3.metric("Failure records", f"{failures:,}")

    st.metric("Dataset failure rate", f"{failures / total * 100:.2f}%")

    st.subheader("Failure modes")
    modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]
    st.bar_chart(ai4i_df[modes].sum())

    st.subheader("Sensor data exploration")
    sensor_columns = [
        "Air temperature",
        "Process temperature",
        "Rotational speed",
        "Torque",
        "Tool wear"
    ]
    selected_sensor = st.selectbox(
        "Choose a sensor to explore",
        sensor_columns
    )
    st.line_chart(ai4i_df[selected_sensor].head(200))

    with st.expander("View dataset sample"):
        st.dataframe(ai4i_df.head(20), use_container_width=True)

    st.caption(
        "This public dataset supports learning and exploration. "
        "Its failure labels are not validated real-world risk probabilities."
    )
else:
    st.error(f"Dataset not found: {DATASET_PATH}")
'''

    app_path.write_text(app_text + analytics_section)
    print("AI4I analytics section added successfully.")
    print("Original SLM and sensor code preserved.")

AI4I analytics section added successfully.
Original SLM and sensor code preserved.


In [67]:
import subprocess
import time
from pathlib import Path

# Stop the existing Streamlit process on port 8501
subprocess.run(
    ["bash", "-lc", "pkill -f 'streamlit run /content/manufacturing_slm/app.py' || true"],
    capture_output=True,
    text=True
)

CompletedProcess(args=['bash', '-lc', "pkill -f 'streamlit run /content/manufacturing_slm/app.py' || true"], returncode=-15, stdout='', stderr='')

In [69]:
!streamlit run /content/manufacturing_slm/app.py \
  --server.port 8501 \
  --server.address 0.0.0.0 \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.enableWebsocketCompression false \
  --server.allowedHosts "*" \
  --browser.gatherUsageStats false

2026-10-14 12:54:35.340 Port 8501 is not available


In [70]:
!ps aux | grep '[s]treamlit'

root       26981  4.0  0.5 525628 71992 ?        Sl   12:53   0:03 /usr/bin/python3 /usr/local/bin/streamlit run /content/manufacturing_slm/app.py --server.port 8501 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false --server.enableWebsocketCompression false --server.allowedHosts * --browser.gatherUsageStats false


In [71]:
!tail -40 /content/manufacturing_slm/streamlit.log

2026-10-14 12:53:33.948 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.171.44.70:8501



In [73]:
from pathlib import Path

app = Path("/content/manufacturing_slm/app.py").read_text()

print("Analytics code saved:", "AI4I Dataset Analytics" in app)
print(
    "Dataset file exists:",
    Path("/content/manufacturing_slm/ai4i2020.csv").exists()
)

Analytics code saved: True
Dataset file exists: True


In [74]:
!streamlit run /content/manufacturing_slm/app.py \
  --server.port 8502 \
  --server.address 0.0.0.0 \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.enableWebsocketCompression false \
  --server.allowedHosts "*" \
  --browser.gatherUsageStats false \
  > /content/manufacturing_slm/streamlit_8502.log 2>&1 &

In [75]:
from IPython.display import display, IFrame
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8502)")
display(IFrame(src=url, width="100%", height=800))

In [76]:
!tail -30 /content/manufacturing_slm/streamlit_8502.log

2026-10-14 12:57:58.526 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.171.44.70:8502

2026-10-14 12:58:29.099 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


In [77]:
import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

Python: 3.13.16
PyTorch: 2.11.0+cpu
GPU available: False
Running on CPU


In [78]:
import torch

print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU available: False
Device: CPU


In [79]:
from pathlib import Path

project = Path("/content/manufacturing_slm")

for name in [
    "ai4i2020.csv",
    "app.py",
    "manufacturing_knowledge.csv"
]:
    path = project / name
    print(f"{name}: {'Found' if path.exists() else 'MISSING'}")

ai4i2020.csv: Found
app.py: Found
manufacturing_knowledge.csv: Found


In [80]:
%pip install -q transformers peft datasets accelerate

In [81]:
import json
from pathlib import Path

project = Path("/content/manufacturing_slm")
output_path = project / "manufacturing_instructions.jsonl"

examples = [
    {
        "instruction": "What should I check when machine vibration is high?",
        "input": "Vibration is above the normal operating range.",
        "output": "High vibration may indicate imbalance, misalignment, loose mounting, or bearing wear. Check the approved vibration limits, inspect mounting and alignment, review recent trends, and follow the plant maintenance procedure."
    },
    {
        "instruction": "What should I do when machine temperature is high?",
        "input": "The machine temperature is increasing.",
        "output": "Check the temperature trend, cooling system, airflow, lubrication, and sensor condition. Compare readings with manufacturer limits. If the temperature exceeds approved limits, follow the plant shutdown and escalation procedure."
    },
    {
        "instruction": "How should I investigate low pressure?",
        "input": "The pressure sensor shows a low reading.",
        "output": "Verify the reading and sensor condition, inspect for leaks or blockages, and check the pump or supply system. Compare the value with the equipment specification before deciding on maintenance."
    },
    {
        "instruction": "How do I investigate voltage fluctuation?",
        "input": "The voltage reading is unstable.",
        "output": "Review voltage trends, inspect approved electrical connections and supply conditions, and check the sensor or measurement equipment. Escalate electrical abnormalities to qualified personnel."
    },
    {
        "instruction": "What does tool wear mean for predictive maintenance?",
        "input": "Tool wear is increasing during operation.",
        "output": "Increasing tool wear can affect product quality and operating load. Compare wear with the tool manufacturer's replacement limits, monitor torque and process quality, and schedule inspection or replacement according to the maintenance plan."
    },
    {
        "instruction": "How should I respond to a possible machine failure?",
        "input": "Sensor readings show an unusual pattern.",
        "output": "Verify the readings, compare them with historical trends and approved limits, and inspect relevant error codes. If there is an unsafe condition, follow the plant's approved stop-work and escalation procedure. Do not rely on a language-model answer alone."
    },
    {
        "instruction": "Explain predictive maintenance simply.",
        "input": "",
        "output": "Predictive maintenance uses equipment data, such as temperature, vibration, pressure, and operating speed, to identify patterns that may indicate a developing problem. It helps maintenance teams plan inspections before an unexpected failure."
    },
    {
        "instruction": "What is the difference between an anomaly and a failure?",
        "input": "",
        "output": "An anomaly is an unusual pattern in equipment data. A failure means the equipment has experienced a defined fault or loss of function. An anomaly can be a warning sign, but it does not always mean a failure will occur."
    }
]

with output_path.open("w", encoding="utf-8") as f:
    for item in examples:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Training examples:", len(examples))
print("Saved to:", output_path)
print("First example:")
print(json.dumps(examples[0], indent=2))

Training examples: 8
Saved to: /content/manufacturing_slm/manufacturing_instructions.jsonl
First example:
{
  "instruction": "What should I check when machine vibration is high?",
  "input": "Vibration is above the normal operating range.",
  "output": "High vibration may indicate imbalance, misalignment, loose mounting, or bearing wear. Check the approved vibration limits, inspect mounting and alignment, review recent trends, and follow the plant maintenance procedure."
}


In [82]:
import json
from pathlib import Path

project = Path("/content/manufacturing_slm")
source_path = project / "manufacturing_instructions.jsonl"
output_path = project / "manufacturing_qwen_train.jsonl"

def format_example(item):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a manufacturing maintenance assistant. "
                "Give clear, cautious, practical explanations. "
                "Do not invent sensor limits or claim certainty."
            )
        },
        {
            "role": "user",
            "content": (
                item["instruction"]
                + (f"\nContext: {item['input']}" if item["input"] else "")
            )
        },
        {
            "role": "assistant",
            "content": item["output"]
        }
    ]
    return {"messages": messages}

with source_path.open(encoding="utf-8") as src:
    examples = [json.loads(line) for line in src if line.strip()]

formatted = [format_example(item) for item in examples]

with output_path.open("w", encoding="utf-8") as dst:
    for item in formatted:
        dst.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Formatted examples:", len(formatted))
print("Saved to:", output_path)
print("\nFirst formatted example:")
print(json.dumps(formatted[0], indent=2, ensure_ascii=False))

Formatted examples: 8
Saved to: /content/manufacturing_slm/manufacturing_qwen_train.jsonl

First formatted example:
{
  "messages": [
    {
      "role": "system",
      "content": "You are a manufacturing maintenance assistant. Give clear, cautious, practical explanations. Do not invent sensor limits or claim certainty."
    },
    {
      "role": "user",
      "content": "What should I check when machine vibration is high?\nContext: Vibration is above the normal operating range."
    },
    {
      "role": "assistant",
      "content": "High vibration may indicate imbalance, misalignment, loose mounting, or bearing wear. Check the approved vibration limits, inspect mounting and alignment, review recent trends, and follow the plant maintenance procedure."
    }
  ]
}


In [83]:
from transformers import AutoTokenizer, AutoConfig

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading model configuration...")
config = AutoConfig.from_pretrained(MODEL_ID)
print("Model type:", config.model_type)

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("Tokenizer loaded:", tokenizer.__class__.__name__)

print("\nChat template available:", tokenizer.chat_template is not None)
print("Vocabulary size:", len(tokenizer))

Loading model configuration...
Model type: qwen2

Loading tokenizer...
Tokenizer loaded: Qwen2Tokenizer

Chat template available: True
Vocabulary size: 151665


In [84]:
import torch
import transformers
import peft
import datasets
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("Device: CPU")

PyTorch: 2.11.0+cpu
Transformers: 5.18.0
PEFT: 0.21.1
Datasets: 4.8.5
Accelerate: 1.15.0
Device: CPU


In [85]:
from pathlib import Path
import json

project = Path("/content/manufacturing_slm")
train_path = project / "manufacturing_qwen_train.jsonl"
adapter_path = project / "qwen_manufacturing_lora"

with train_path.open(encoding="utf-8") as f:
    training_examples = [json.loads(line) for line in f if line.strip()]

print("Training examples:", len(training_examples))
print("Training file exists:", train_path.exists())
print("Adapter will be saved to:", adapter_path)
print("Planned method: LoRA, CPU, float32")
print("Original Qwen model will remain unchanged.")

Training examples: 8
Training file exists: True
Adapter will be saved to: /content/manufacturing_slm/qwen_manufacturing_lora
Planned method: LoRA, CPU, float32
Original Qwen model will remain unchanged.


In [87]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [88]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading Qwen model on CPU...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

model.config.use_cache = False

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

Loading Qwen model on CPU...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 270,336 || all params: 494,303,104 || trainable%: 0.0547


In [89]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 270,336 || all params: 494,303,104 || trainable%: 0.0547


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [90]:
model.print_trainable_parameters()

trainable params: 270,336 || all params: 494,303,104 || trainable%: 0.0547


In [91]:
print("LoRA attached:", hasattr(model, "peft_config"))

if hasattr(model, "peft_config"):
    model.print_trainable_parameters()
else:
    print("LoRA is not attached yet. We need to retry the setup.")

LoRA attached: True
trainable params: 270,336 || all params: 494,303,104 || trainable%: 0.0547


In [92]:
import json
from pathlib import Path

project = Path("/content/manufacturing_slm")
train_path = project / "manufacturing_qwen_train.jsonl"

with train_path.open(encoding="utf-8") as f:
    training_examples = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

def format_for_training(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

training_texts = [
    format_for_training(example)
    for example in training_examples
]

print("Training examples:", len(training_texts))
print("First formatted example:")
print(training_texts[0])

Training examples: 8
First formatted example:
<|im_start|>system
You are a manufacturing maintenance assistant. Give clear, cautious, practical explanations. Do not invent sensor limits or claim certainty.<|im_end|>
<|im_start|>user
What should I check when machine vibration is high?
Context: Vibration is above the normal operating range.<|im_end|>
<|im_start|>assistant
High vibration may indicate imbalance, misalignment, loose mounting, or bearing wear. Check the approved vibration limits, inspect mounting and alignment, review recent trends, and follow the plant maintenance procedure.<|im_end|>



In [93]:
import torch
from pathlib import Path

# CPU training settings
device = torch.device("cpu")
model.to(device)
model.train()

# Tokenize the formatted examples
encoded_examples = [
    tokenizer(
        text,
        truncation=True,
        max_length=384,
        return_tensors="pt"
    )
    for text in training_texts
]

optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=2e-4
)

epochs = 3
print("Starting LoRA training on CPU...")

for epoch in range(epochs):
    total_loss = 0.0

    for item in encoded_examples:
        input_ids = item["input_ids"].to(device)
        attention_mask = item["attention_mask"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=input_ids
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{epochs} — "
        f"Average loss: {total_loss / len(encoded_examples):.4f}"
    )

print("Training loop completed.")

Starting LoRA training on CPU...
Epoch 1/3 — Average loss: 4.2830
Epoch 2/3 — Average loss: 3.7788
Epoch 3/3 — Average loss: 3.3956
Training loop completed.


In [94]:
from pathlib import Path

adapter_path = Path("/content/manufacturing_slm/qwen_manufacturing_lora")
adapter_path.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(adapter_path))
tokenizer.save_pretrained(str(adapter_path))

print("Fine-tuned LoRA adapter saved successfully!")
print("Saved location:", adapter_path)
print("Files:", [p.name for p in adapter_path.iterdir()])

Fine-tuned LoRA adapter saved successfully!
Saved location: /content/manufacturing_slm/qwen_manufacturing_lora
Files: ['chat_template.jinja', 'tokenizer.json', 'adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'README.md']


In [97]:
import torch

model.eval()

messages = [
    {
        "role": "system",
        "content": "You are a manufacturing maintenance assistant. Give safe, practical guidance and follow approved plant procedures."
    },
    {
        "role": "user",
        "content": "What should we check when a machine has high vibration?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

input_ids = inputs["input_ids"].to("cpu")
attention_mask = inputs["attention_mask"].to("cpu")

with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=120,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

new_tokens = outputs[0][input_ids.shape[-1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Fine-tuned SLM response:")
print(answer)

Fine-tuned SLM response:
Check the bearing condition of the motor or gearbox; if necessary, replace it with new parts.


In [98]:
import pandas as pd
from pathlib import Path

data_path = Path("/content/manufacturing_slm/ai4i2020.csv")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMachine failure counts:")
print(df["Machine failure"].value_counts())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (10000, 12)

Columns:
['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

Machine failure counts:
Machine failure
0    9661
1     339
Name: count, dtype: int64

First 5 rows:


,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [99]:
from sklearn.model_selection import train_test_split

# Sensor and machine-type inputs
feature_columns = [
    "Type",
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear"
]

X = df[feature_columns].copy()
y = df["Machine failure"].copy()

# Split before training; preserve the failure/normal ratio
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nTraining target counts:")
print(y_train.value_counts())
print("\nTesting target counts:")
print(y_test.value_counts())

Training rows: 8000
Testing rows: 2000

Training target counts:
Machine failure
0    7729
1     271
Name: count, dtype: int64

Testing target counts:
Machine failure
0    1932
1      68
Name: count, dtype: int64


In [100]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Convert machine Type (L, M, H) into numeric features
preprocessor = ColumnTransformer(
    transformers=[
        ("type", OneHotEncoder(handle_unknown="ignore"), ["Type"]),
        ("numbers", "passthrough", [
            "Air temperature",
            "Process temperature",
            "Rotational speed",
            "Torque",
            "Tool wear"
        ])
    ]
)

# Build the classifier
failure_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Train
failure_model.fit(X_train, y_train)

# Evaluate on unseen test data
y_pred = failure_model.predict(X_test)

print("Model training completed!")
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Normal", "Failure"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Model training completed!

Classification Report:
              precision    recall  f1-score   support

      Normal       0.98      1.00      0.99      1932
     Failure       0.97      0.46      0.62        68

    accuracy                           0.98      2000
   macro avg       0.97      0.73      0.81      2000
weighted avg       0.98      0.98      0.98      2000

Confusion Matrix:
[[1931    1]
 [  37   31]]


In [101]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Get failure probabilities from the trained model
failure_probabilities = failure_model.predict_proba(X_test)[:, 1]

# Try a lower threshold than the default 0.50
threshold = 0.25
y_pred_tuned = (failure_probabilities >= threshold).astype(int)

print("Decision threshold:", threshold)
print("\nUpdated Classification Report:")
print(classification_report(
    y_test,
    y_pred_tuned,
    target_names=["Normal", "Failure"],
    zero_division=0
))

print("Updated Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

Decision threshold: 0.25

Updated Classification Report:
              precision    recall  f1-score   support

      Normal       0.99      0.99      0.99      1932
     Failure       0.69      0.75      0.72        68

    accuracy                           0.98      2000
   macro avg       0.84      0.87      0.85      2000
weighted avg       0.98      0.98      0.98      2000

Updated Confusion Matrix:
[[1909   23]
 [  17   51]]


In [102]:
import joblib
from pathlib import Path

model_path = Path("/content/manufacturing_slm/machine_failure_model.joblib")

joblib.dump(failure_model, model_path)

print("Machine-failure model saved!")
print("Location:", model_path)
print("File exists:", model_path.exists())

Machine-failure model saved!
Location: /content/manufacturing_slm/machine_failure_model.joblib
File exists: True


In [103]:
from pathlib import Path
import shutil

project = Path("/content/manufacturing_slm")
app_file = project / "app.py"
backup_file = project / "app_before_failure_prediction.py"

if app_file.exists():
    shutil.copy2(app_file, backup_file)
    print("Backup created successfully!")
    print("Backup path:", backup_file)
    print("Backup exists:", backup_file.exists())
else:
    print("ERROR: app.py was not found.")

Backup created successfully!
Backup path: /content/manufacturing_slm/app_before_failure_prediction.py
Backup exists: True


In [104]:
from pathlib import Path

app_file = Path("/content/manufacturing_slm/app.py")
lines = app_file.read_text(encoding="utf-8").splitlines()

print("Total lines:", len(lines))
print("\n--- Last 50 lines of app.py ---\n")

for number, line in enumerate(lines[-50:], start=max(1, len(lines) - 49)):
    print(f"{number}: {line}")

Total lines: 253

--- Last 50 lines of app.py ---

204: st.divider()
205: st.header("📊 AI4I Dataset Analytics")
206: 
207: DATASET_PATH = PROJECT / "ai4i2020.csv"
208: 
209: if DATASET_PATH.exists():
210:     @st.cache_data
211:     def load_ai4i_data(path):
212:         return pd.read_csv(path)
213: 
214:     ai4i_df = load_ai4i_data(str(DATASET_PATH))
215: 
216:     total = len(ai4i_df)
217:     failures = int(ai4i_df["Machine failure"].sum())
218:     normal = total - failures
219: 
220:     col1, col2, col3 = st.columns(3)
221:     col1.metric("Total records", f"{total:,}")
222:     col2.metric("Normal records", f"{normal:,}")
223:     col3.metric("Failure records", f"{failures:,}")
224: 
225:     st.metric("Dataset failure rate", f"{failures / total * 100:.2f}%")
226: 
227:     st.subheader("Failure modes")
228:     modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]
229:     st.bar_chart(ai4i_df[modes].sum())
230: 
231:     st.subheader("Sensor data exploration")
232:     sensor_columns 

In [105]:
from pathlib import Path

app_file = Path("/content/manufacturing_slm/app.py")
code = r'''

# ============================================================
# MACHINE FAILURE PREDICTION
# ============================================================
import joblib

st.divider()
st.header("🔧 Machine Failure Prediction")
st.write(
    "Enter the current machine readings to estimate failure risk. "
    "This is a learning prototype, not a certified safety system."
)

MODEL_PATH = PROJECT / "machine_failure_model.joblib"

if MODEL_PATH.exists():
    @st.cache_resource
    def load_failure_model(path):
        return joblib.load(path)

    failure_model = load_failure_model(str(MODEL_PATH))

    with st.form("failure_prediction_form"):
        machine_type = st.selectbox("Machine type", ["L", "M", "H"])
        air_temp = st.number_input(
            "Air temperature (K)", min_value=250.0,
            max_value=350.0, value=298.1, step=0.1
        )
        process_temp = st.number_input(
            "Process temperature (K)", min_value=250.0,
            max_value=400.0, value=308.6, step=0.1
        )
        rotational_speed = st.number_input(
            "Rotational speed (rpm)", min_value=0,
            max_value=5000, value=1500
        )
        torque = st.number_input(
            "Torque (Nm)", min_value=0.0,
            max_value=200.0, value=40.0, step=0.1
        )
        tool_wear = st.number_input(
            "Tool wear (minutes)", min_value=0,
            max_value=300, value=10
        )

        submitted = st.form_submit_button("Predict machine failure risk")

    if submitted:
        import pandas as pd

        input_data = pd.DataFrame([{
            "Type": machine_type,
            "Air temperature": air_temp,
            "Process temperature": process_temp,
            "Rotational speed": rotational_speed,
            "Torque": torque,
            "Tool wear": tool_wear
        }])

        probability = float(failure_model.predict_proba(input_data)[0, 1])
        threshold = 0.25

        st.metric("Model-estimated failure probability", f"{probability:.1%}")

        if probability >= threshold:
            st.warning(
                "Elevated model-estimated risk. Verify readings and follow "
                "your approved plant inspection and escalation procedures."
            )
        else:
            st.success(
                "Below the prototype's alert threshold. This does not guarantee "
                "that the machine is safe or will not fail."
            )

        st.caption(
            "The 25% alert threshold is an experimental setting selected on a "
            "held-out test set. Results may differ on real machines."
        )
else:
    st.error(f"Failure prediction model not found: {MODEL_PATH}")
'''

with app_file.open("a", encoding="utf-8") as f:
    f.write(code)

print("Prediction section appended:", app_file.exists())
print("Current line count:", len(app_file.read_text(encoding="utf-8").splitlines()))

Prediction section appended: True
Current line count: 335


In [106]:
import ast
from pathlib import Path

app_file = Path("/content/manufacturing_slm/app.py")
source_code = app_file.read_text(encoding="utf-8")

try:
    ast.parse(source_code)
    print("Syntax check passed! ✅")
except SyntaxError as e:
    print("Syntax error:", e)
    print("Line:", e.lineno)

Syntax check passed! ✅


In [107]:
import subprocess
import time
from pathlib import Path

project = Path("/content/manufacturing_slm")

# Stop the previous Streamlit process on port 8502
subprocess.run(
    "pkill -f 'streamlit run /content/manufacturing_slm/app.py.*8502'",
    shell=True,
    capture_output=True
)

CompletedProcess(args="pkill -f 'streamlit run /content/manufacturing_slm/app.py.*8502'", returncode=-15, stdout=b'', stderr=b'')

In [109]:
!streamlit run /content/manufacturing_slm/app.py --server.port 8502 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false --server.enableWebsocketCompression false --server.allowedHosts "*" --browser.gatherUsageStats false > /content/manufacturing_slm/streamlit_8502.log 2>&1 &

In [110]:
from IPython.display import display, IFrame
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8502)")
display(IFrame(src=url, width="100%", height=800))

In [111]:
from pathlib import Path

app_file = Path("/content/manufacturing_slm/app.py")
lines = app_file.read_text(encoding="utf-8").splitlines()

print("Model-loading and chat-related lines:\n")

keywords = [
    "AutoModelForCausalLM",
    "AutoTokenizer",
    "from_pretrained",
    "generate(",
    "def load",
    "def get",
    "st.chat_input",
    "st.chat_message",
    "model ="
]

for i, line in enumerate(lines, start=1):
    if any(keyword in line for keyword in keywords):
        start = max(1, i - 3)
        end = min(len(lines), i + 5)
        print(f"\n--- Lines {start}–{end} ---")
        for j in range(start, end + 1):
            print(f"{j}: {lines[j - 1]}")

Model-loading and chat-related lines:


--- Lines 4–12 ---
4: import torch
5: from pathlib import Path
6: from PIL import Image
7: from transformers import AutoTokenizer, AutoModelForCausalLM
8: 
9: PROJECT = Path("/content/manufacturing_slm")
10: CSV_FILE = PROJECT / "manufacturing_knowledge.csv"
11: MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
12: 

--- Lines 20–28 ---
20: )
21: 
22: @st.cache_resource
23: def load_model():
24:     tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
25:     model = AutoModelForCausalLM.from_pretrained(
26:         MODEL_ID, torch_dtype=torch.float32
27:     )
28:     model.to("cpu")

--- Lines 21–29 ---
21: 
22: @st.cache_resource
23: def load_model():
24:     tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
25:     model = AutoModelForCausalLM.from_pretrained(
26:         MODEL_ID, torch_dtype=torch.float32
27:     )
28:     model.to("cpu")
29:     model.eval()

--- Lines 22–30 ---
22: @st.cache_resource
23: def load_model():
24:     tokenizer = AutoT

In [112]:
from pathlib import Path
import shutil

project = Path("/content/manufacturing_slm")
app_file = project / "app.py"
backup_file = project / "app_before_lora_integration.py"

shutil.copy2(app_file, backup_file)

print("Backup created:", backup_file)
print("Backup exists:", backup_file.exists())

Backup created: /content/manufacturing_slm/app_before_lora_integration.py
Backup exists: True


In [113]:
from pathlib import Path

app_file = Path("/content/manufacturing_slm/app.py")
source = app_file.read_text(encoding="utf-8")

old_function = '''@st.cache_resource
def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float32
    )
    model.to("cpu")
    model.eval()
    return tokenizer, model
'''

new_function = '''@st.cache_resource
def load_model():
    from peft import PeftModel

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float32
    )
    base_model.to("cpu")

    adapter_path = PROJECT / "qwen_manufacturing_lora"

    if adapter_path.exists() and (adapter_path / "adapter_config.json").exists():
        model = PeftModel.from_pretrained(
            base_model,
            str(adapter_path)
        )
    else:
        model = base_model

    model.to("cpu")
    model.eval()
    return tokenizer, model
'''

if old_function not in source:
    print("STOP: Exact original function not found. No changes made.")
else:
    app_file.write_text(
        source.replace(old_function, new_function, 1),
        encoding="utf-8"
    )
    print("Model-loading function updated successfully.")

Model-loading function updated successfully.


In [114]:
import ast
from pathlib import Path

project = Path("/content/manufacturing_slm")
app_file = project / "app.py"
adapter_path = project / "qwen_manufacturing_lora"

# Check Python syntax
try:
    ast.parse(app_file.read_text(encoding="utf-8"))
    print("App syntax: PASS")
except SyntaxError as e:
    print(f"App syntax: ERROR on line {e.lineno}: {e.msg}")

# Check saved LoRA adapter
config_file = adapter_path / "adapter_config.json"
print("Adapter folder exists:", adapter_path.exists())
print("Adapter config exists:", config_file.exists())

App syntax: PASS
Adapter folder exists: True
Adapter config exists: True


In [115]:
import subprocess
import time
from pathlib import Path

# Stop Streamlit processes running this app on port 8502
subprocess.run(
    ["bash", "-lc",
     "pids=$(ps -eo pid,args | grep '[s]treamlit run /content/manufacturing_slm/app.py' | grep '8502' | awk '{print $1}'); "
     "if [ -n \"$pids\" ]; then kill $pids; fi"],
    capture_output=True,
    text=True
)

time.sleep(2)

# Start the updated app
log_path = "/content/manufacturing_slm/streamlit_8502.log"
log_file = open(log_path, "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "/content/manufacturing_slm/app.py",
        "--server.port", "8502",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.enableWebsocketCompression", "false",
        "--server.allowedHosts", "*",
        "--browser.gatherUsageStats", "false"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

time.sleep(5)
print("Streamlit process started with PID:", process.pid)
print("Log file:", log_path)
print(Path(log_path).read_text(errors="replace")[-2500:])

Streamlit process started with PID: 40860
Log file: /content/manufacturing_slm/streamlit_8502.log



In [117]:
from pathlib import Path
import subprocess

log_path = Path("/content/manufacturing_slm/streamlit_8502.log")

print("=== Streamlit log ===")
if log_path.exists():
    print(log_path.read_text(errors="replace")[-3000:] or "Log is empty.")
else:
    print("Log file not found.")

print("\n=== Port 8502 ===")
result = subprocess.run(
    ["curl", "--max-time", "3", "-sS", "-o", "/dev/null",
     "-w", "HTTP status: %{http_code}", "http://127.0.0.1:8502/"],
    capture_output=True,
    text=True
)
print(result.stdout or result.stderr)

=== Streamlit log ===
2026-10-14 13:50:22.884 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.171.44.70:8502



=== Port 8502 ===
HTTP status: 200


In [118]:
from IPython.display import display, IFrame
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8502)")
print("Fresh app URL:", url)

display(IFrame(src=url, width="100%", height=800))

Fresh app URL: https://8502-m-s-kkb-usc1b0-1g0z8sf68cxcc-b.us-central1-0.prod.colab.dev
